In [2]:
#Importing dependencies 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("/users/imbahndu/Desktop/Columbia DBM/Typing dataset/lightbgm.csv")

In [4]:
df

,Unnamed: 0,Hand,HoldTime,Direction,LatencyTime,FlightTime,Parkinsons,Gender_Male
0,0,0,183.6,0,339.8,199.2,False,False
1,3,0,66.4,2,250.0,171.9,True,True
2,4,1,187.5,3,234.4,31.3,True,False
3,5,1,89.8,1,226.6,113.3,False,True
4,6,1,93.8,1,460.9,367.2,True,True
...,...,...,...,...,...,...,...,...
64431,86788,0,175.8,0,31.3,113.3,True,True
64432,86790,0,78.1,0,234.4,156.3,True,True
64433,86791,0,46.9,0,515.6,453.1,True,True
64434,86792,1,85.9,1,218.8,46.9,False,False


In [5]:
df.isnull().sum()

Unnamed: 0     0
Hand           0
HoldTime       0
Direction      0
LatencyTime    0
FlightTime     0
Parkinsons     0
Gender_Male    0
dtype: int64

In [6]:
df["Parkinsons"].value_counts()


Parkinsons
False    32481
True     31955
Name: count, dtype: int64

In [8]:
#Dataset was already preprocessed and balanced, so jump right into modeling 

#defining labels

y = df["Parkinsons"]
X = df.drop(columns = ["Parkinsons", "Hand", "Direction", "Unnamed: 0"])

from imblearn.over_sampling import SMOTE
from collections import Counter

# Assume X and y are your features and labels
print("Original class distribution:", Counter(y))

rus = SMOTE(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled class distribution:", Counter(y_resampled))

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, random_state = 42, test_size = 0.1)

Original class distribution: Counter({False: 32481, True: 31955})


/users/imbahndu/.local/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Resampled class distribution: Counter({False: 32481, True: 32481})


In [9]:
print(X_train)

print(y_train)

       HoldTime  LatencyTime  FlightTime  Gender_Male
57672     105.5        457.0       328.1         True
55525      62.5        468.8       406.3         True
60212      85.9        316.4       230.5        False
26033      23.4        226.6       257.8         True
45817     125.0        218.8       140.6         True
...         ...          ...         ...          ...
62570     101.6        562.5       265.6         True
38158     128.9        253.9       132.8        False
860        85.9        113.3        31.3        False
15795      74.2        300.8       203.1         True
56422     142.9        235.0        74.0        False

[58465 rows x 4 columns]
57672    False
55525     True
60212    False
26033     True
45817    False
         ...  
62570    False
38158    False
860      False
15795     True
56422    False
Name: Parkinsons, Length: 58465, dtype: bool


In [11]:
#training model
model_xgb = XGBClassifier()

model_xgb.fit(X_train, y_train)

pred = model_xgb.predict(X_test)

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc_score = accuracy_score(y_test, pred)

acc_score

0.6027397260273972

In [12]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

       False       0.60      0.64      0.62      3289
        True       0.60      0.57      0.59      3208

    accuracy                           0.60      6497
   macro avg       0.60      0.60      0.60      6497
weighted avg       0.60      0.60      0.60      6497



In [ ]:
#Class is so freaking imbalanced--> will adjust this demain using sklearn.utils resample 

#This is the best i could get so keeping this and try sklearn.utils demain and also maybe try SMOTE and RandomOverSampler on the entire dataset. Do this for both tyoing and this dataset